# SP-CMDNet (RSNA single-cohort edition)
### A dual-view, scanner-invariant ViT with tissue-superposition disentanglement

This notebook ports the **portable ideas** from the SP-CMDNet methodology onto the single-cohort **RSNA Breast Cancer Detection** dataset your base code already uses. It is an *honest* contribution: every module here is trainable on RSNA alone, with no 3D data and no external masks.

**What we take from SP-CMDNet (and can actually run on RSNA):**

| SP-CMDNet component | How it maps to RSNA |
|---|---|
| **Cross-View Strip Attention** (Comp. D) | RSNA has CC + MLO per breast → pair them and cross-attend within radial strips |
| **DHTD de-occlusion block** (Comp. C) | Dual-branch *lesion vs. background-superposition* decomposition + gated subtraction + **orthogonality loss** — used self-supervised (no 3D teacher needed in the forward pass) |
| **Domain-Adversarial GRL** (L_adv) | Use RSNA `machine_id` as the domain label → scanner-invariant features |
| **Class-balanced Focal loss** (L_cls) | Replaces the base `BCELoss`; handles the severe cancer imbalance |
| **Patient-level splitting** | Fixes a data-leakage bug in the base code (see §1) |
| Otsu crop + orientation flip + CLAHE + resize | Harmonization / preprocessing (§2) |

**What is intentionally NOT here** (requires data outside RSNA, so claiming it would be dishonest):
* the 2.5D **DBT teacher**, the **synthetic-2D projection**, and the feature-distillation loss `L_feat` → need **BCS-DBT** volumes;
* the mask-guided spatial loss `L_spatial` → needs **CBIS-DDSM** pixel masks.

The architecture below is deliberately built as the **2D student** of SP-CMDNet, so if you later obtain BCS-DBT + CBIS-DDSM you can attach the teacher and the two missing losses without redesigning anything.

# 0. Imports

In [ ]:
import os

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("device:", device)

# 1. Fixing the data-leakage bug (patient-level split)

The base notebook splits like this:

```python
train_df, val_df = train_test_split(df, test_size=0.25, stratify=df['cancer'])
```

RSNA has **multiple images per patient** (CC + MLO, left + right). A row-level split therefore puts the *same patient* in both train and test, so the model can memorise patient-specific texture and your reported AUC is optimistically biased. Reviewers call this **patient-level data leakage**, and it is the single most common reason medical-imaging papers get rejected.

We split on **`patient_id`** with `GroupShuffleSplit` so no patient crosses the boundary.

In [ ]:
df = pd.read_csv("/kaggle/input/rsna-breast-cancer-detection/train.csv")

# --- patient-level split (no patient in both train and test) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(gss.split(df, df["cancer"], groups=df["patient_id"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

# sanity: zero patient overlap
overlap = set(train_df["patient_id"]) & set(val_df["patient_id"])
print("patients  train:", train_df['patient_id'].nunique(),
      " val:", val_df['patient_id'].nunique(),
      " overlap:", len(overlap))

# --- machine_id -> contiguous domain index (fit on the full df so val maps cleanly) ---
machines = sorted(df["machine_id"].unique())
machine_to_domain = {m: i for i, m in enumerate(machines)}
N_DOMAINS = len(machines)
print("scanners (domains):", N_DOMAINS)

# 2. Preprocessing & harmonization

Per the methodology: Otsu breast cropping, orientation flipping (right breasts flipped so the chest wall is always on the left), CLAHE, and aspect-aware resizing. Pectoral-muscle suppression on MLO views is provided as an **optional** step (a lightweight approximation of the paper's Hough/U-Net approach) — left off by default so it can't silently corrupt data.

In [ ]:
def otsu_breast_crop(img):
    """Binarize, keep the largest connected component (the breast), crop to its bbox."""
    img8 = (255 * (img / (img.max() + 1e-6))).astype("uint8")
    _, mask = cv2.threshold(img8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return img
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    ys, xs = np.where(labels == largest)
    return img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]


def suppress_pectoral_mlo(img):
    """
    OPTIONAL, lightweight pectoral suppression for MLO views.
    Assumes the breast has already been oriented so the chest wall is on the LEFT,
    i.e. the pectoral triangle sits in the TOP-LEFT corner. This is an approximation
    of the paper's Hough/U-Net method -- validate before trusting it.
    """
    h, w = img.shape
    img8 = (255 * (img / (img.max() + 1e-6))).astype("uint8")
    # bright region in the top-left is the candidate pectoral muscle
    thr = np.percentile(img8, 80)
    tri = np.zeros_like(img8)
    pts = np.array([[0, 0], [int(0.35 * w), 0], [0, int(0.45 * h)]], np.int32)
    cv2.fillConvexPoly(tri, pts, 255)
    pect = (img8 > thr) & (tri > 0)
    out = img.copy()
    out[pect] = 0.0
    return out


def preprocess(img, laterality, view, img_size=(512, 512), do_pectoral=False):
    img = otsu_breast_crop(img)

    # orientation: flip right breasts so chest wall is always on the left border
    if laterality == "R":
        img = cv2.flip(img, 1)

    if do_pectoral and view == "MLO":
        img = suppress_pectoral_mlo(img)

    # CLAHE on an 8-bit view, then back to float
    img8 = (255 * (img / (img.max() + 1e-6))).astype("uint8")
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img8 = clahe.apply(img8)

    img = cv2.resize(img8, (img_size[1], img_size[0]))   # cv2 wants (W, H)
    img = img.astype("float32") / 255.0
    return img

# 3. Dual-view dataset

We group each breast by **(`patient_id`, `laterality`)** and pair its **CC** and **MLO** views into a single training sample. If a breast is missing one view, we duplicate the available one. The label is the breast-level `cancer`, and the domain label is the breast's `machine_id`.

> Images are loaded on CPU and moved to the GPU in the training loop (not inside `__getitem__`), which keeps `DataLoader` workers usable.

In [ ]:
def build_breast_samples(frame):
    """One sample per breast: a CC image_id, an MLO image_id, label, domain."""
    samples = []
    for (pid, lat), g in frame.groupby(["patient_id", "laterality"]):
        cc = g[g["view"] == "CC"]
        mlo = g[g["view"] == "MLO"]
        cc_id = int(cc["image_id"].iloc[0]) if len(cc) else None
        mlo_id = int(mlo["image_id"].iloc[0]) if len(mlo) else None
        if cc_id is None and mlo_id is None:
            continue
        cc_id = cc_id if cc_id is not None else mlo_id      # duplicate if a view is missing
        mlo_id = mlo_id if mlo_id is not None else cc_id
        samples.append({
            "patient_id": int(pid),
            "laterality": lat,
            "cc_id": cc_id,
            "mlo_id": mlo_id,
            "label": int(g["cancer"].max()),
            "domain": machine_to_domain[g["machine_id"].iloc[0]],
        })
    return pd.DataFrame(samples)


train_breasts = build_breast_samples(train_df)
val_breasts = build_breast_samples(val_df)
print("breasts  train:", len(train_breasts), " val:", len(val_breasts))
print("cancer rate  train:", train_breasts["label"].mean().round(4),
      " val:", val_breasts["label"].mean().round(4))

In [ ]:
class DualViewRSNA(Dataset):
    def __init__(self, breasts, img_path, img_size=(512, 512), do_pectoral=False):
        self.breasts = breasts.reset_index(drop=True)
        self.img_path = img_path
        self.img_size = img_size
        self.do_pectoral = do_pectoral

    def __len__(self):
        return len(self.breasts)

    def _load(self, pid, image_id, lat, view):
        f = os.path.join(self.img_path, f"{pid}_{image_id}.png")
        img = cv2.imread(f, cv2.IMREAD_GRAYSCALE).astype("float32")
        img = preprocess(img, lat, view, self.img_size, self.do_pectoral)
        return torch.tensor(img[np.newaxis])          # (1, H, W)

    def __getitem__(self, idx):
        r = self.breasts.iloc[idx]
        cc = self._load(r["patient_id"], r["cc_id"], r["laterality"], "CC")
        mlo = self._load(r["patient_id"], r["mlo_id"], r["laterality"], "MLO")
        y = torch.tensor([r["label"]], dtype=torch.float32)
        d = torch.tensor(r["domain"], dtype=torch.long)
        return cc, mlo, y, d

In [ ]:
img_path = "/kaggle/input/rsna-breast-cancer-512-pngs"
img_size = (512, 512)
batch_size = 8
train_samples = 1000
val_samples = 500

train_ds = DualViewRSNA(train_breasts, img_path, img_size, do_pectoral=False)
val_ds = DualViewRSNA(val_breasts, img_path, img_size, do_pectoral=False)

# class-balanced sampling at the breast level
counts = train_breasts["label"].value_counts()
w = train_breasts["label"].apply(lambda x: 1.0 / counts[x]).values
train_sampler = WeightedRandomSampler(w, train_samples, replacement=True)

train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=train_sampler, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)

# 4. Architecture

Five pieces, each tied to a line in the methodology:

1. **`SharedEncoder`** — a Siamese CNN→Transformer hybrid (your fusion backbone) producing a `C×H×W` feature map per view.
2. **`DHTD`** — Depth-Hallucination & Tissue-Deocclusion block: lesion/occlusion decomposition, gated subtraction, pseudo-depth cross-attention, + an orthogonality penalty.
3. **`CrossViewStripAttention`** — radial strip-wise cross-attention between CC and MLO.
4. **`GradReverse` + domain head** — scanner-invariance via a Gradient Reversal Layer over `machine_id`.
5. **classification head** on the fused bilateral representation.

## 4.1 Shared Siamese encoder (CNN + Transformer)

In [ ]:
class SharedEncoder(nn.Module):
    """
    Siamese encoder: a strided-conv stem (local inductive bias) followed by a few
    transformer layers (global context), returning a spatial feature map.

    input:  (b, 1, 512, 512)   ->   output: (b, d_model, 16, 16)
    """
    def __init__(self, n_channels, d_model, nhead, dim_ff, n_layers):
        super().__init__()

        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, stride=2, padding=1),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

        chans = [n_channels, 32, 64, 128, 256, d_model]      # 512 -> 16 (5 strides)
        self.stem = nn.Sequential(*[block(chans[i], chans[i + 1]) for i in range(5)])

        self.grid = 16
        self.pos = nn.Parameter(torch.randn(self.grid * self.grid, d_model))
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_ff,
                                           activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)

    def forward(self, x):
        f = self.stem(x)                              # (b, C, 16, 16)
        b, c, h, w = f.shape
        t = f.flatten(2).transpose(1, 2)             # (b, HW, C)
        t = self.encoder(t + self.pos)
        return t.transpose(1, 2).reshape(b, c, h, w)  # (b, C, 16, 16)

## 4.2 DHTD — Depth-Hallucination & Tissue-Deocclusion

The paper's primary architectural contribution. On RSNA we keep the *module* (which needs no 3D data at inference) and its **orthogonality constraint**, and we drop only the teacher-supervised `L_feat` term. We adapt the pseudo-depth attention to be **spatially preserving** (spatial positions query `K` learnable virtual-depth planes) so the `H×W` map survives for the downstream strip attention.

In [ ]:
class DHTD(nn.Module):
    def __init__(self, d_model, n_depth=5, nhead=4):
        super().__init__()
        self.to_lesion = nn.Conv2d(d_model, d_model, 1)
        self.to_occ = nn.Conv2d(d_model, d_model, 1)
        self.gate = nn.Conv2d(d_model, d_model, 3, padding=1)

        self.depth_emb = nn.Parameter(torch.randn(n_depth, d_model))   # virtual slice planes
        self.depth_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, f):
        # --- dual-branch decomposition ---
        lesion = self.to_lesion(f)                       # focal density / spiculation
        occ = self.to_occ(f)                             # overlapping fibroglandular tissue
        g = torch.sigmoid(self.gate(occ))
        deocc = lesion - g * occ                         # gated de-occlusion subtraction

        # --- pseudo-depth cross-attention (spatial queries, depth keys/values) ---
        b, c, h, w = deocc.shape
        q = deocc.flatten(2).transpose(1, 2)             # (b, HW, C)
        kv = self.depth_emb.unsqueeze(0).expand(b, -1, -1)  # (b, K, C)
        att, _ = self.depth_attn(q, kv, kv)              # (b, HW, C)
        q = self.norm(q + att)
        refined = q.transpose(1, 2).reshape(b, c, h, w)

        # --- orthogonality: keep lesion and occlusion subspaces distinct ---
        lv = lesion.flatten(1)
        ov = occ.flatten(1)
        orth = F.cosine_similarity(lv, ov, dim=1).abs().mean()
        return refined, orth

## 4.3 Cross-View Strip Attention

A lesion lies at roughly the same radial distance from the nipple in CC and MLO. We split each feature map into `n_strips` horizontal radial bands and cross-attend CC↔MLO **within the same band**, reinforcing bilateral consistency without forcing a full dense match.

In [ ]:
class CrossViewStripAttention(nn.Module):
    def __init__(self, d_model, nhead=4, n_strips=4):
        super().__init__()
        self.n_strips = n_strips
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)

    def _to_strips(self, f):
        b, c, h, w = f.shape
        s = self.n_strips
        sh = h // s
        f = f[:, :, :sh * s, :].reshape(b, c, s, sh, w)      # (b, C, s, sh, w)
        f = f.permute(0, 2, 3, 4, 1).reshape(b * s, sh * w, c)  # (b*s, tokens, C)
        return f, (b, c, s, sh, w)

    def forward(self, query_view, context_view):
        q, shp = self._to_strips(query_view)
        kv, _ = self._to_strips(context_view)
        out, _ = self.attn(q, kv, kv)                         # CC strip attends MLO strip
        b, c, s, sh, w = shp
        out = out.reshape(b, s, sh, w, c).permute(0, 4, 1, 2, 3).reshape(b, c, s * sh, w)
        return out

## 4.4 Gradient Reversal Layer (scanner invariance)

In [ ]:
class GradReverse(Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad):
        return -ctx.lambd * grad, None


def grad_reverse(x, lambd=1.0):
    return GradReverse.apply(x, lambd)

## 4.5 Full model

In [ ]:
class SPCMDNetStudent(nn.Module):
    """
    2D dual-view student of SP-CMDNet, trainable end-to-end on RSNA.
    Returns (classification logit, domain logit, orthogonality penalty).
    """
    def __init__(self, d_model=256, nhead=4, dim_ff=512, enc_layers=2,
                 n_depth=5, n_strips=4, n_domains=10, head_hidden=128):
        super().__init__()
        self.encoder = SharedEncoder(1, d_model, nhead, dim_ff, enc_layers)   # shared weights
        self.dhtd = DHTD(d_model, n_depth, nhead)
        self.cross = CrossViewStripAttention(d_model, nhead, n_strips)

        self.cls_head = nn.Sequential(
            nn.Linear(2 * d_model, head_hidden), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(head_hidden, 1))
        self.dom_head = nn.Sequential(
            nn.Linear(2 * d_model, head_hidden), nn.ReLU(inplace=True),
            nn.Linear(head_hidden, n_domains))

    def _branch(self, x):
        f = self.encoder(x)
        f, orth = self.dhtd(f)
        return f, orth

    def forward(self, cc, mlo, lambd=1.0):
        f_cc, o_cc = self._branch(cc)
        f_mlo, o_mlo = self._branch(mlo)

        # bilateral strip cross-attention (residual)
        f_cc = f_cc + self.cross(f_cc, f_mlo)
        f_mlo = f_mlo + self.cross(f_mlo, f_cc)

        g = torch.cat([f_cc.mean((2, 3)), f_mlo.mean((2, 3))], dim=1)   # (b, 2C)

        logit = self.cls_head(g)                       # (b, 1)
        dom_logit = self.dom_head(grad_reverse(g, lambd))
        return logit, dom_logit, (o_cc + o_mlo)

### Sanity check

In [ ]:
_m = SPCMDNetStudent(n_domains=N_DOMAINS).to(device)
_cc = torch.randn(2, 1, 512, 512, device=device)
_mlo = torch.randn(2, 1, 512, 512, device=device)
with torch.no_grad():
    _logit, _dom, _orth = _m(_cc, _mlo, lambd=1.0)
print("cls logit:", tuple(_logit.shape), "| domain logit:", tuple(_dom.shape),
      "| orth:", round(float(_orth), 4))
print("params:", round(sum(p.numel() for p in _m.parameters()) / 1e6, 2), "M")
del _m, _cc, _mlo

# 5. Composite loss

On RSNA we keep the three loss terms that need no extra data:

* **`L_cls`** — class-balanced (focal) BCE **with logits** (numerically stable; fixes the base code's `Sigmoid+BCELoss`).
* **`L_adv`** — cross-entropy over `machine_id` through the GRL.
* **`L_orth`** — the DHTD orthogonality penalty.

`L_total = L_cls + λ_adv · L_adv + λ_orth · L_orth`. The two SP-CMDNet terms that need external data (`L_feat`, `L_spatial`) are left as documented hooks for later.

In [ ]:
def focal_bce_with_logits(logits, targets, alpha=0.75, gamma=2.0):
    """Class-balanced focal loss on logits."""
    p = torch.sigmoid(logits)
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = p * targets + (1 - p) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    loss = alpha_t * (1 - p_t) ** gamma * ce
    return loss.mean()


lambda_adv = 0.1
lambda_orth = 0.1

# 6. Training & evaluation

A plain PyTorch loop (the multi-output model + GRL schedule don't fit the base `create_supervised_trainer` cleanly). The GRL weight `λ` ramps from 0→1 over training, the standard DANN schedule. We report the metrics that matter for imbalanced screening: **AUC-ROC, AUC-PR, Sensitivity@90%-Specificity**, best-threshold **F1**, and **ECE** (calibration).

In [ ]:
def sensitivity_at_specificity(y, p, target_spec=0.90):
    fpr, tpr, _ = roc_curve(y, p)
    spec = 1 - fpr
    ok = spec >= target_spec
    return float(tpr[ok].max()) if ok.any() else 0.0


def best_f1(y, p):
    y = np.asarray(y); p = np.asarray(p)
    ts = np.linspace(0.05, 0.95, 19)
    best = 0.0
    for t in ts:
        pred = (p >= t).astype(int)
        tp = ((pred == 1) & (y == 1)).sum()
        fp = ((pred == 1) & (y == 0)).sum()
        fn = ((pred == 0) & (y == 1)).sum()
        f1 = 2 * tp / (2 * tp + fp + fn + 1e-9)
        best = max(best, f1)
    return float(best)


def expected_calibration_error(y, p, n_bins=10):
    y = np.asarray(y); p = np.asarray(p)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        m = (p > bins[i]) & (p <= bins[i + 1])
        if m.sum() == 0:
            continue
        ece += (m.sum() / len(p)) * abs(y[m].mean() - p[m].mean())
    return float(ece)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    for cc, mlo, y, d in loader:
        cc, mlo = cc.to(device), mlo.to(device)
        logit, _, _ = model(cc, mlo, lambd=0.0)
        ps.append(torch.sigmoid(logit).cpu().numpy().ravel())
        ys.append(y.numpy().ravel())
    y = np.concatenate(ys); p = np.concatenate(ps)
    return {
        "AUC-ROC": roc_auc_score(y, p) if len(np.unique(y)) > 1 else float("nan"),
        "AUC-PR": average_precision_score(y, p) if len(np.unique(y)) > 1 else float("nan"),
        "Sens@90Spec": sensitivity_at_specificity(y, p),
        "F1": best_f1(y, p),
        "ECE": expected_calibration_error(y, p),
    }

In [ ]:
model = SPCMDNetStudent(
    d_model=256, nhead=4, dim_ff=512, enc_layers=2,
    n_depth=5, n_strips=4, n_domains=N_DOMAINS,
).to(device)

optimizer = Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

max_epochs = 5
best_auc = 0.0

for epoch in range(1, max_epochs + 1):
    model.train()
    running = 0.0
    for it, (cc, mlo, y, d) in enumerate(train_loader):
        cc, mlo, y, d = cc.to(device), mlo.to(device), y.to(device), d.to(device)

        # DANN-style GRL ramp 0 -> 1
        p = ((epoch - 1) * len(train_loader) + it) / (max_epochs * len(train_loader))
        lambd = 2.0 / (1.0 + np.exp(-10 * p)) - 1.0

        logit, dom_logit, orth = model(cc, mlo, lambd=lambd)

        l_cls = focal_bce_with_logits(logit, y)
        l_adv = F.cross_entropy(dom_logit, d)
        loss = l_cls + lambda_adv * l_adv + lambda_orth * orth

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item()

    metrics = evaluate(model, val_loader)
    if not np.isnan(metrics["AUC-ROC"]) and metrics["AUC-ROC"] > best_auc:
        best_auc = metrics["AUC-ROC"]
        torch.save(model.state_dict(), "best_sp_cmdnet_student.pt")

    msg = " | ".join(f"{k}: {v:.3f}" for k, v in metrics.items())
    print(f"Epoch {epoch}  train_loss: {running / len(train_loader):.3f}  ||  {msg}")

# 7. Ablation study scaffold

The methodology asks for a module-by-module ablation. Because every block here is a constructor flag or a loss weight, you can reproduce the SP-CMDNet ablation table on RSNA by toggling:

| Variant | How to switch it off |
|---|---|
| Baseline (single-view ViT) | feed the same view as CC and MLO; `n_strips`→ cross-attn becomes a near-identity |
| + Cross-View Strip Attention | compare with the step above |
| + DHTD de-occlusion | set `lambda_orth = 0` **and** bypass `self.dhtd` (return `f, 0`) |
| + Domain GRL | set `lambda_adv = 0` |
| **Full model** | all on |

Report each row with the §6 metrics on the **patient-level** test split. Keep the random seed fixed and run 3 seeds for mean ± std — reviewers expect variance bars.

---

## Honest contribution statement (for your paper)

> *We adapt three components of the SP-CMDNet framework to a single-cohort 2D setting: (i) a tissue-superposition disentanglement block with an orthogonality constraint, (ii) radial cross-view (CC/MLO) strip attention, and (iii) domain-adversarial scanner-invariance over acquisition hardware. We do **not** claim 3D→2D knowledge distillation, as paired tomosynthesis data is outside this cohort; the architecture is constructed as the 2D student so the distillation stage can be added when BCS-DBT becomes available.*

This is defensible at review. Claiming the full "cross-modality distillation" on RSNA alone would not be.

## If you later get BCS-DBT + CBIS-DDSM
1. Train the 2.5D teacher (5-slice slab, Z-axis cross-slice attention) on BCS-DBT.
2. Generate the synthetic-2D projection to make spatially-aligned 3D↔2D pairs.
3. Add `L_feat` (feature distillation: this student's DHTD output ↔ frozen teacher map) and `L_spatial` (Dice on CBIS-DDSM masks / KL to the teacher heatmap).
4. The `SPCMDNetStudent` above is already the target — only the two losses and the teacher are new.